<a href="https://colab.research.google.com/github/QUANHONGLE/CS421-emotion-prediction/blob/main/Q3_BERT_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install torch
!pip install pandas
!pip install numpy
!pip install scikit-learn
!pip install sympy --upgrade

In [ ]:
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, classification_report
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

# Load data
train_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_train.csv')
test_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_test.csv', on_bad_lines='skip')
dev_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_dev.csv', on_bad_lines='skip')

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Dev:", dev_df.shape)

In [ ]:
from transformers import AutoTokenizer

# Load pretrained BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Test it works
sample = tokenizer("Hello, how are you?", truncation=True, padding='max_length', max_length=128, return_tensors='pt')
print("Tokenizer loaded!")
print("Input IDs shape:", sample['input_ids'].shape)

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128, is_test=False):
        self.tokenizer = tokenizer
        self.texts = df['text'].astype(str).tolist()
        self.max_length = max_length
        self.is_test = is_test
        if not is_test:
            self.emotion = torch.tensor(df['Emotion'].values, dtype=torch.float32)
            self.polarity = torch.tensor(df['EmotionalPolarity'].values, dtype=torch.long)
            self.empathy = torch.tensor(df['Empathy'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0)
        }
        if not self.is_test:
            item['emotion'] = self.emotion[idx]
            item['polarity'] = self.polarity[idx]
            item['empathy'] = self.empathy[idx]
        return item

# Create datasets
train_ds = EmotionDataset(train_df, tokenizer)
dev_ds   = EmotionDataset(dev_df, tokenizer)
test_ds  = EmotionDataset(test_df, tokenizer, is_test=True)

# Create dataloaders
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
dev_dl   = DataLoader(dev_ds, batch_size=16)
test_dl  = DataLoader(test_ds, batch_size=16)

print("Datasets ready!")
print(f"Train: {len(train_ds)}, Dev: {len(dev_ds)}, Test: {len(test_ds)}")

In [ ]:
from transformers import AutoModel

class BERTEmotionModel(nn.Module):
    def __init__(self, model_name='bert-base-uncased', dropout=0.3, num_polarity_classes=4):
        super(BERTEmotionModel, self).__init__()

        # Load pretrained BERT
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size  # 768 for bert-base

        self.dropout = nn.Dropout(dropout)

        # Task-specific heads
        self.emotion_head  = nn.Linear(hidden_size, 1)
        self.polarity_head = nn.Linear(hidden_size, num_polarity_classes)
        self.empathy_head  = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token
        cls_output = self.dropout(cls_output)

        emotion  = self.emotion_head(cls_output).squeeze(1)
        polarity = self.polarity_head(cls_output)
        empathy  = self.empathy_head(cls_output).squeeze(1)
        return emotion, polarity, empathy

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

model = BERTEmotionModel()
model = model.to(device)
print("Model ready!")

In [ ]:
from transformers import get_linear_schedule_with_warmup

def train_bert(model, train_dl, dev_dl, epochs=3, lr=2e-5):
    criterion_reg = nn.MSELoss()
    criterion_cls = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    total_steps = len(train_dl) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    for epoch in range(epochs):
        # Training
        model.train()
        total_loss = 0
        for batch in train_dl:
            input_ids     = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            emotion       = batch['emotion'].to(device)
            polarity      = batch['polarity'].to(device)
            empathy       = batch['empathy'].to(device)

            optimizer.zero_grad()
            pred_emotion, pred_polarity, pred_empathy = model(input_ids, attention_mask)

            loss = criterion_reg(pred_emotion, emotion) + \
                   criterion_cls(pred_polarity, polarity) + \
                   criterion_reg(pred_empathy, empathy)

            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        # Evaluation
        model.eval()
        all_emotion, all_polarity, all_empathy = [], [], []
        true_emotion, true_polarity, true_empathy = [], [], []

        with torch.no_grad():
            for batch in dev_dl:
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)

                pred_emotion, pred_polarity, pred_empathy = model(input_ids, attention_mask)

                all_emotion.extend(pred_emotion.cpu().numpy())
                all_polarity.extend(torch.argmax(pred_polarity, dim=1).cpu().numpy())
                all_empathy.extend(pred_empathy.cpu().numpy())

                true_emotion.extend(batch['emotion'].numpy())
                true_polarity.extend(batch['polarity'].numpy())
                true_empathy.extend(batch['empathy'].numpy())

        mae_emotion = mean_absolute_error(true_emotion, all_emotion)
        mae_empathy = mean_absolute_error(true_empathy, all_empathy)
        report      = classification_report(true_polarity, all_polarity, zero_division=0, output_dict=True)
        f1          = report['macro avg']['f1-score']

        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss:.3f} | "
              f"MAE Emotion: {mae_emotion:.3f} | MAE Empathy: {mae_empathy:.3f} | F1 Polarity: {f1:.3f}")

    return model

print("Starting BERT training...")
model = train_bert(model, train_dl, dev_dl)

In [ ]:
def predict(model, test_dl):
    model.eval()
    all_emotion, all_polarity, all_empathy = [], [], []
    with torch.no_grad():
        for batch in test_dl:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            pred_emotion, pred_polarity, pred_empathy = model(input_ids, attention_mask)

            all_emotion.extend(pred_emotion.cpu().numpy())
            all_polarity.extend(torch.argmax(pred_polarity, dim=1).cpu().numpy())
            all_empathy.extend(pred_empathy.cpu().numpy())

    return all_emotion, all_polarity, all_empathy

emotion_preds, polarity_preds, empathy_preds = predict(model, test_dl)

predictions_df = pd.DataFrame({
    'id': test_df['id'].values,
    'Emotion': emotion_preds,
    'EmotionalPolarity': polarity_preds,
    'Empathy': empathy_preds
})

predictions_df.to_csv('predictions_bert.csv', index=False)
print("Saved predictions_bert.csv!")
print(predictions_df.head())